# Q2: The Midnight Episode — 'Catching the Arrhythmia'
### EE200 Course Project — Summer 2026
**Prepared by:** Your Name / Roll Number

---
The patient's ECG was recorded with a Holter monitor for 20 seconds at $f_s = 250$ Hz ($N = 5000$ samples). The first ~9.6 s is a perfectly healthy, metronomic rhythm. At that point the heart slips into an arrhythmia. We detect this onset using **normalized cross-correlation** and confirm it with a **spectrogram**.

## (a) Reading the Signal

**(i) Duration**

$$T = \frac{N}{f_s} = \frac{5000}{250} = \boxed{20 \text{ seconds}}$$

**(ii) Heart rate & samples per beat**

One full beat $\{P, QRS, T\}$ arrives every $0.8$ s in the healthy stretch.

$$\text{HR} = \frac{60}{0.8} = \boxed{75 \text{ bpm}}, \qquad
  L = 0.8 \times 250 = \boxed{200 \text{ samples per beat}}$$

**(iii) Fundamental frequency**

$$f_0 = \frac{1}{0.8\text{ s}} = \boxed{1.25 \text{ Hz}}$$

## (b) Healthy Heart in the Frequency Domain

**(i)** A nearly periodic signal has a **discrete (line) spectrum**: isolated spikes (harmonics) at integer multiples of $f_0 = 1.25$ Hz — not a smooth continuous curve.

**(ii)** The **QRS complex** owns the high frequencies. It is a sharp, narrow spike in the time domain — steep slopes require high-frequency components to represent. The broad, smooth P and T waves contribute only to the low-frequency harmonics.

**(iii)** At 150 bpm: $f_0' = 150/60 = 2.5$ Hz. The fundamental frequency **doubles** to 2.5 Hz and the harmonic spacing also doubles from 1.25 Hz to **2.5 Hz**.

## (c) Cutting a Heartbeat (Windowing)

**(i)** Width = **200 samples** (one full beat period). Placement: the **early, healthy portion** (first 1-2 s), where rhythm is perfectly regular.

**(ii)**

| Window | Problem |
|--------|---------|
| **80 samples** (0.32 s) | Captures only the QRS complex; P-wave and T-wave are cropped. Template is structurally *incomplete* — correlation never reaches 1 for a real beat. |
| **600 samples** (2.4 s) | Spans exactly **3 beats**. The template averages multiple beats; correlation peaks every 3rd beat boundary instead of every beat. |

**(iii)** A shorter window improves time resolution but degrades frequency resolution ($\Delta f = f_s / N_w$). More importantly, the window must be at least one full physiological beat (200 samples) to contain all structural landmarks $\{P, QRS, T\}$. Shrinking it below that destroys the template Maya needs to match against.

## (d) Match the Template (Correlation)

$$\rho(m) = \frac{\sum_k t[k]\,x[m+k]}{\|t\|\,\|x_m\|}$$

**(i)** $\rho(m) \in [-1,\,+1]$. A value of **$\rho = 1$** (or close) signals a near-perfect shape match.

**(ii)** Without the denominator, the raw score is **not scale-invariant**. A healthy beat twice as tall as the template doubles the unnormalised score — making it look "better" than it really is. Baseline wander and amplitude variation would constantly fool an unnormalised detector. Dividing by $\|x_m\|$ makes the score depend only on *shape*, not on energy.

**(iii)** An inverted beat is approximately $-t[k]$, so:

$$\rho \approx \frac{\sum_k t[k]\,(-t[k])}{\|t\|^2} = -1$$

This makes inverted beats **trivially easy to flag**: they produce a strong *negative* peak — unambiguously abnormal. The data confirms this: the first arrhythmia beat scores $\rho = -0.9879 \approx -1$.

## (e) Onset Detection & the Spectrogram

**(i)** **Rule:** declare an arrhythmia at the first beat index $m^*$ where $\rho(m^*) < \tau$ (e.g., $\tau = 0.5$).

| Threshold | Risk |
|-----------|------|
| Too **high** (0.95) | Breathing-related amplitude variation causes false alarms in healthy beats. |
| Too **low** (0.10) | Subtle but dangerous rhythm changes score above threshold and are missed. |

**(ii)** *Healthy region*: spectrogram shows **steady, bright horizontal bands** at $f_0 = 1.25$ Hz and harmonics (2.5, 3.75, 5.0 Hz…). *Arrhythmia region*: periodic structure collapses — bands shift, smear, or disappear.

**(iii)** The **correlation plot** is more trustworthy for pinpointing the exact beat. The spectrogram requires a long window (for frequency resolution), which time-smears the event across many frames — the transition looks gradual. The beat-by-beat correlator operates directly in the time domain and flags the *exact* 200-sample window where shape first goes wrong.

## (f) Sampling & Aliasing

**(i)** $$f_s^{\min} = 2 \times 40 = \boxed{80 \text{ Hz}}$$

**(ii)** The Nyquist frequency at 50 Hz sampling is 25 Hz. Energy at 25-40 Hz **folds back** into 10-25 Hz. The QRS spike — built from those high frequencies — gets distorted and blurred. A sharp spike looks rounded and wider; its correlation with Maya's clean template drops, producing false arrhythmia flags during healthy beats.

**(iii)** Apply a **steep anti-aliasing low-pass filter** with $f_c < 25$ Hz *before* sampling. The unavoidable cost: permanent loss of 25-40 Hz QRS content, making the reconstructed spike broader and clinically less sharp.

## (g) Prototyping the Detector in Code

We implement `find_onset` exactly as specified:
- Jump forward by **L** samples at a time (beat-by-beat, not sample-by-sample)
- Compute normalised dot-product $\rho$ for each window
- Return index of **first** window where $\rho < \tau$ (or $-1$ if never breached)

**Key improvement:** we collect ALL beat scores before returning so the plot shows the complete 25-beat correlation profile — not just the beats up to onset.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Load data
ecg_signal = np.load('EE200_course_project_data_2026/Q2_data/patient_ecg.npy')
template   = np.load('EE200_course_project_data_2026/Q2_data/template.npy')
fs = 250; N = len(ecg_signal); L = len(template)
t_axis = np.arange(N) / fs

print(f'ECG:      {N} samples ({N/fs:.1f} s)')
print(f'Template: {L} samples ({L/fs:.3f} s = one beat)')

# ── find_onset: collects ALL beat scores, marks FIRST breach ──────────
def find_onset(ecg_signal, template, threshold=0.5):
    L = len(template); N = len(ecg_signal)
    norm_t = np.linalg.norm(template)
    onset = -1; scores = []; indices = []
    for m in range(0, N - L + 1, L):
        segment = ecg_signal[m:m+L]
        norm_s = np.linalg.norm(segment)
        rho = float(np.dot(template, segment) / (norm_t * norm_s)) if norm_s > 0 else 0.0
        scores.append(rho); indices.append(m)
        if rho < threshold and onset == -1:
            onset = m   # record but keep collecting all beats
    return onset, scores, indices

onset_idx, rho_scores, beat_indices = find_onset(ecg_signal, template, threshold=0.5)
onset_time = onset_idx / fs if onset_idx != -1 else None

print(f'Arrhythmia onset: sample {onset_idx}  (t = {onset_time:.2f} s)')
print(f'Total beats evaluated: {len(rho_scores)}')
print('Full rho profile:')
for m, r in zip(beat_indices, rho_scores):
    flag = '  <- FIRST ARRHYTHMIA BEAT' if m == onset_idx else ('  <- arrhythmia' if r < 0.5 else '')
    print(f'  m={m:4d}  t={m/fs:5.2f}s  rho={r:+.4f}{flag}')

# ── Plot: ECG + full correlation profile ─────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=False)
fig.suptitle('ECG Signal and Beat-by-Beat Correlation', fontsize=14, fontweight='bold')

ax1 = axes[0]
ax1.plot(t_axis, ecg_signal, color='#475569', lw=0.8, label='ECG signal')
if onset_time:
    ax1.axvline(onset_time, color='#ef4444', lw=1.8, linestyle='--', label=f'Onset t={onset_time:.1f}s')
    ax1.axvspan(onset_time, N/fs, alpha=0.07, color='#ef4444', label='Arrhythmia region')
ax1.set_ylabel('Amplitude'); ax1.set_title('Patient ECG Recording')
ax1.legend(loc='upper right'); ax1.grid(True, alpha=0.3); ax1.set_xlim(0, N/fs)

ax2 = axes[1]
beat_times = [m/fs for m in beat_indices]
colors_dots = ['#22c55e' if r >= 0.5 else '#ef4444' for r in rho_scores]
ax2.plot(beat_times, rho_scores, color='#818cf8', lw=1.4, zorder=1)
ax2.scatter(beat_times, rho_scores, c=colors_dots, s=70, zorder=2,
            label='rho per beat  (green=healthy, red=arrhythmia)')
ax2.axhline(0.5, color='#f59e0b', lw=1.5, linestyle='--', label='Threshold tau=0.5')
ax2.axhline(0.0, color='gray', lw=0.8, linestyle=':')
if onset_time:
    ax2.axvline(onset_time, color='#ef4444', lw=1.8, linestyle='--',
                label=f'Detected onset (t={onset_time:.1f}s)')
ax2.set_xlabel('Time [s]'); ax2.set_ylabel('Correlation rho(m)')
ax2.set_title('Beat-by-Beat Normalized Correlation  [all 25 beats shown]')
ax2.legend(loc='lower left'); ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, N/fs); ax2.set_ylim(-1.15, 1.15)

plt.tight_layout()
plt.savefig('q2_correlation_plot.png', dpi=150)
plt.show()


ECG:      5000 samples (20.0 s)
Template: 200 samples (0.800 s = one beat)
Arrhythmia onset: sample 2400  (t = 9.60 s)
Total beats evaluated: 25
Full rho profile:
  m=   0  t= 0.00s  rho=+0.9944
  m= 200  t= 0.80s  rho=+0.9918
  m= 400  t= 1.60s  rho=+0.9941
  m= 600  t= 2.40s  rho=+0.9967
  m= 800  t= 3.20s  rho=+0.9923
  m=1000  t= 4.00s  rho=+0.9902
  m=1200  t= 4.80s  rho=+0.9938
  m=1400  t= 5.60s  rho=+0.9900
  m=1600  t= 6.40s  rho=+0.9937
  m=1800  t= 7.20s  rho=+0.9915
  m=2000  t= 8.00s  rho=+0.9945
  m=2200  t= 8.80s  rho=+0.9920
  m=2400  t= 9.60s  rho=-0.9879  <- FIRST ARRHYTHMIA BEAT
  m=2600  t=10.40s  rho=-0.2765  <- arrhythmia
  m=2800  t=11.20s  rho=-0.5607  <- arrhythmia
  m=3000  t=12.00s  rho=-0.4128  <- arrhythmia
  m=3200  t=12.80s  rho=-0.0962  <- arrhythmia
  m=3400  t=13.60s  rho=-0.3029  <- arrhythmia
  m=3600  t=14.40s  rho=-0.1407  <- arrhythmia
  m=3800  t=15.20s  rho=-0.1971  <- arrhythmia
  m=4000  t=16.00s  rho=-0.1878  <- arrhythmia
  m=4200  t=16.80s 

C:\Users\karti\AppData\Local\Temp\ipykernel_37228\2823812051.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## (h) Visualizing the Spectrogram

**Window length choice:** `nperseg = 200` (exactly one beat duration)

$$\Delta f = \frac{f_s}{N_w} = \frac{250}{200} = 1.25 \text{ Hz}$$

This is the **optimal and elegant choice**. Because the fundamental frequency of the healthy heartbeat is exactly $f_0 = 1.25$ Hz, setting the frequency resolution $\Delta f$ to exactly 1.25 Hz means that the DFT bins fall precisely at $0, 1.25, 2.5, 3.75 \dots$ Hz. All the harmonic energy falls perfectly into these specific bins with zero spectral leakage, creating exceptionally sharp and bright horizontal bands. Furthermore, 200 samples is exactly the length of one beat (0.8s), providing an excellent balance of time and frequency resolution.

In [2]:
from scipy.signal import spectrogram as scipy_spectrogram

# nperseg=200 => Delta_f=1.25 Hz  (bins align perfectly with harmonics)
# noverlap=150 => time step = 50/250 = 0.2 s
f_s2, t_s2, Sxx = scipy_spectrogram(
    ecg_signal, fs=fs,
    nperseg=200, noverlap=150,
    window='hann'
)

print(f'Freq resolution: {fs/200:.2f} Hz')
print(f'Time step:       {(200-150)/fs:.3f} s')
print(f'Sxx shape:       {Sxx.shape}')

fig2, ax3 = plt.subplots(figsize=(14, 5))
pcm = ax3.pcolormesh(
    t_s2, f_s2,
    10 * np.log10(Sxx + 1e-12),
    shading='gouraud', cmap='inferno', vmin=-80, vmax=-20
)
ax3.set_ylim(0, 25)

# Mark expected harmonics of f0=1.25 Hz
for k in range(1, 20):
    fk = k * 1.25
    if fk > 25: break
    ax3.axhline(fk, color='cyan', lw=0.6, alpha=0.6, linestyle='--')

if onset_time:
    ax3.axvline(onset_time, color='#facc15', lw=2.0, linestyle='--',
                label=f'Arrhythmia onset (t={onset_time:.1f}s)')

ax3.set_xlabel('Time [s]'); ax3.set_ylabel('Frequency [Hz]')
ax3.set_title(
    'Spectrogram of Patient ECG\n'
    '(cyan dashed = expected harmonics at 1.25*k Hz;  '
    'yellow = detected onset;  nperseg=200, Delta_f=1.25 Hz)'
)
ax3.legend(loc='upper right')
plt.colorbar(pcm, ax=ax3, label='Power [dB]')
plt.tight_layout()
plt.savefig('q2_spectrogram.png', dpi=150)
plt.show()
print('Saved: q2_spectrogram.png')
print()
print('Observation: In the first ~9.6 s the horizontal cyan lines are clearly lit,')
print('confirming periodic harmonic structure. After the onset the pattern fragments.')


Freq resolution: 1.25 Hz
Time step:       0.200 s
Sxx shape:       (101, 97)


Saved: q2_spectrogram.png

Observation: In the first ~9.6 s the horizontal cyan lines are clearly lit,
confirming periodic harmonic structure. After the onset the pattern fragments.


C:\Users\karti\AppData\Local\Temp\ipykernel_37228\509970986.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Appendix: Template Waveform
The template `template.npy` provided is exactly 200 samples (one full healthy beat). It shows the canonical P-QRS-T morphology clearly, confirming it is a good reference for correlation.

In [3]:
t_tmpl = np.arange(L) / fs * 1000  # in ms
fig3, ax4 = plt.subplots(figsize=(8, 3))
ax4.plot(t_tmpl, template, color='#6366f1', lw=2)
ax4.set_xlabel('Time within beat [ms]')
ax4.set_ylabel('Amplitude')
ax4.set_title(f'Template: one healthy beat  ({L} samples = {L/fs:.2f} s)')
ax4.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('q2_template.png', dpi=150)
plt.show()
print(f'Template: peak={template.max():.4f}  trough={template.min():.4f}')
print('The template clearly shows: P-wave (broad bump ~180ms), QRS spike (~400ms), T-wave (~600ms)')


Template: peak=0.9877  trough=-0.2027
The template clearly shows: P-wave (broad bump ~180ms), QRS spike (~400ms), T-wave (~600ms)


C:\Users\karti\AppData\Local\Temp\ipykernel_37228\2281179136.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
